# 第五轮：向全量70%准确率推进

**结论：未通过升级门槛，保留第四轮概率推荐，冻结推荐为 `previous_probability_recommendation`。** 当前推荐 **1,696/2,588（65.53%）**，距离70%至少还需净增加 **116场**正确预测。 本轮最好的已观察2023准确率是 `accuracy_grid_blend` 的 **66.27%**（1,715场正确），距目标还差97场；这个排序不替代冻结选择。

主实验比较 **75个新配置和3个旧配置**，使用326项候选输入、33,833场训练目标和原样保留的2,588场2023目标。不同模型使用对应输入子集。目标70%需要至少 **1,812场**正确；没有挑高置信比赛、减少分母或放宽T−21历史规则。市场信息属于另一个信息条件，不混入本表。

## 为什么改变方法，以及如何选择

用户授权本轮改为**准确率优先**。跨级别历史补充Tour以外已发生的对局，技术组成细化已有发球统计；学习器比较有限年龄交互、随机傅里叶残差及logistic/squared-hinge目标。原概率推荐保持独立，不因新目标被改名或覆盖。公式、来源、反证和局限见[本轮协议](docs/research/round5_protocol.md)及[学习器说明](docs/research/round5_learning.md)。换损失不保证更准，校准也不能改变固定0.5阈值下的胜负决策。

2020–2021先选accuracy最高的候选；log loss必须不超过第四轮参照+0.005，同accuracy再比较log loss及固定名称。每种方法取一个合格配置，从至多四个候选枚举步长0.1的非负、和为1权重，包含单模型顶点。早期锁定的是：`r5_rff512_g2_squared_hinge_C0.03` × 0.5、`r5_rff512_g2_logistic_C0.03` × 0.3、`multilevel_linear_C0.1_halfNone` × 0.2。

在2022门槛年，参考accuracy为 **67.13%**，候选为 **67.33%**，变化 **+0.198个百分点**；需至少提高 **0.5个百分点**。候选log loss为 **0.58992**，参考为 **0.59466**，允许恶化不超过 **0.005**。两项条件必须同时满足；这不是显著性检验。

全部2020–2022折的候选重选结果为：`r5_rff512_g2_squared_hinge_C0.03` × 0.5、`r5_rff512_g2_logistic_C0.03` × 0.2、`both_spline_C0.01_half5.0` × 0.3。若升级失败，它只作为探索性对照；正式推荐仍为旧模型。全部选择和权重在本轮2023评分前冻结。**2022与2023此前已被查看，本轮仍是后续探索，2024–2025最终测试保持关闭。** 三个旧候选的纳入也继承此前探索历史，不能称为独立预注册。

## 同一批比赛的结果

| 模型 / 角色 | 正确场次 | 准确率 ↑ | log loss ↓ | 距70%差额 |
| --- | ---: | ---: | ---: | ---: |
| `rank_reference`（探索性对照） | 1,640 | 63.37% | 0.62980 | 172 |
| `previous_probability_recommendation`（本轮正式推荐） | 1,696 | 65.53% | 0.60941 | 116 |
| `previous_accuracy_comparator`（探索性对照） | 1,705 | 65.88% | 0.61057 | 107 |
| `previous_exploratory_best_loss`（探索性对照） | 1,693 | 65.42% | 0.60868 | 119 |
| `r5_rff512_g2_squared_hinge_C0.03`（探索性对照） | 1,698 | 65.61% | 0.61489 | 114 |
| `accuracy_grid_blend`（训练期候选，未获升级） | 1,715 | 66.27% | 0.60987 | 97 |
| `all_inputs_spline_C0.01`（2023最低观察loss，探索性） | 1,706 | 65.92% | 0.60645 | 106 |

训练期选出的本轮候选 `accuracy_grid_blend` 相对第四轮模型，纠正 **98场**旧错、同时新增 **79场**错误，净变化 **+19场**，accuracy变化 **+0.734个百分点**。赛事开始日期分块的配对区间为 **[-0.157, +1.638]个百分点**。**准确率区间跨过0，不能确认优于第四轮推荐。** 区间还没有涵盖跨日期重复球员依赖和全部模型搜索。

候选log-loss改善（旧减新）为 **-0.00046**，对应条件区间 **[-0.00543, +0.00463]**。负数表示概率预测变差；准确率提升不能替代概率质量报告。完整表另外保留Brier分数。

本轮保存比较中，最低2023 log loss来自 `all_inputs_spline_C0.01`：**0.60645**，准确率 **65.92%**（1,706场正确）。它是值得保留的概率预测对照，但这个已观察排序不改变本轮accuracy门槛或正式推荐。

![第五轮全量准确率与概率损失](results/round5/round5_comparison.png)

## 哪些信息和目标有帮助

以下是严格固定学习器、正则和近期权重时，加入信息后的训练期变化；范围展示多个设置，不是置信区间：

- **加入跨级别历史 / 线性**：6组同设置配对，pooled CV准确率变化 **[+0.277, +0.701]个百分点**；其中6组提高。
- **加入技术组成 / 线性**：6组同设置配对，pooled CV准确率变化 **[-0.179, +0.277]个百分点**；其中3组提高。
- **同时加入两组 / 线性**：6组同设置配对，pooled CV准确率变化 **[+0.326, +0.701]个百分点**；其中6组提高。
- **加入跨级别历史 / 样条**：3组同设置配对，pooled CV准确率变化 **[+0.179, +0.343]个百分点**；其中3组提高。
- **加入技术组成 / 样条**：3组同设置配对，pooled CV准确率变化 **[+0.065, +0.196]个百分点**；其中3组提高。
- **同时加入两组 / 样条**：3组同设置配对，pooled CV准确率变化 **[+0.326, +0.392]个百分点**；其中3组提高。

相同表示和C下，squared hinge相对logistic的12项配对，pooled CV准确率变化范围为 **[-0.294, +0.506]个百分点**，6项提高。各项log-loss变化和逐年方向同时保留，不能用单个最优设置证明某损失天然更优。

Notebook保存全部 **45项信息添加/删除对照**和 **12项学习目标对照**。删除比较以“删除后减删除前”的accuracy变化及“删除前减删除后”的loss变化计，正数表示删除后更好。相关输入的增量是条件预测比较，不能解释成因果贡献。家族赢家还经过重新选择参数，不等同于严格同参数消融。所有候选CV都保留；未满足概率护栏的家族不会被强行列为正式候选。

## 复跑与交接

```bash
.venv/bin/python -m scripts.train_round5
.venv/bin/python -m scripts.report_round5
```

[执行notebook](round5.ipynb)、[HTML](round5.html)、[本轮协议](docs/research/round5_protocol.md)和独立的模型文件均保留此前实验。`search_registry.json`记录全部候选，`temporal_cv.csv`记录训练期比较，`frozen_selection.json`记录门槛与权重，`validation_predictions.csv`保存逐场概率，`frozen_models.joblib`用于冻结推理。

最终测试应比较本轮冻结推荐、第四轮概率推荐和既有强对照，使用相同信息时点和完整人群；在那之前，不将重复使用2023上的最高准确率写成已确认的70%泛化能力或SOTA。


## 完整模型表

正式推荐由冻结门槛决定，2023排序只是探索性观察。

In [1]:
from IPython.display import display
import pandas as pd
from scripts.report_round5 import load_results,comparison_table,family_table,matched_information_controls,matched_objective_controls
data=load_results()
pd.set_option('display.max_columns',None)
pd.set_option('display.max_rows',None)
pd.set_option('display.precision',6)
display(comparison_table(data))

,model,description,recommended,matches,correct,accuracy,mean_log_loss,brier,correct_short_of_70,CV_accuracy,CV_log_loss
0,accuracy_grid_blend,Accuracy-selected convex blend,False,2588,1715,0.662674,0.609871,0.211658,97,0.671941,0.599059
1,multilevel_spline_C0.03_half3,Add cross-level history · spline · C=0.03 / h=3.0,False,2588,1711,0.661128,0.608014,0.210930,101,0.665742,0.598557
2,r5_rff512_g2_logistic_C0.03,RFF 512 / g=2 · logistic · C=0.03,False,2588,1708,0.659969,0.608984,0.211322,104,0.668515,0.598554
3,style_linear_C0.1_half3.0,Add technical composition · linear · C=0.1 / h...,False,2588,1708,0.659969,0.610013,0.211812,104,0.663622,0.604761
4,r5_context_squared_hinge_C0.03,context · squared hinge · C=0.03,False,2588,1707,0.659583,0.611118,0.212218,105,0.666232,0.599535
5,all_inputs_spline_C0.01,All old and new information,False,2588,1706,0.659196,0.606453,0.210323,106,0.665742,0.599153
6,previous_accuracy_comparator,Round 3 expanded linear comparator,False,2588,1705,0.658810,0.610569,0.212039,107,0.658401,0.608397
7,r5_existing_odd_spline_C0.01_context1_half3.0,Existing spline control · C=0.01,False,2588,1704,0.658423,0.607506,0.210770,108,0.665742,0.598764
8,both_linear_C0.1_half3.0,Add both information groups · linear · C=0.1 /...,False,2588,1703,0.658037,0.608254,0.211152,109,0.667047,0.599516
9,r5_linear_squared_hinge_C0.003,linear · squared hinge · C=0.003,False,2588,1700,0.656878,0.610241,0.211927,112,0.664437,0.599246


## 全部同设置信息与学习目标对照

所有变化统一以正数表示候选更好；保留逐年方向，不将模型相关性解释为因果。

In [2]:
display(matched_information_controls(data))
display(matched_objective_controls(data))

,comparison,learner,information,reference,candidate,pooled_accuracy_gain,pooled_loss_reduction,2020_accuracy_gain,2020_loss_reduction,2021_accuracy_gain,2021_loss_reduction,2022_accuracy_gain,2022_loss_reduction
0,add,linear,multilevel,core_linear_C0.001_halfNone,multilevel_linear_C0.001_halfNone,0.002773,0.004522,0.008237,0.000466,0.003769,0.005407,-0.000791,0.005635
1,add,linear,style,core_linear_C0.001_halfNone,style_linear_C0.001_halfNone,-0.001794,0.000484,-0.004942,0.000189,-0.000419,0.000540,-0.001582,0.000574
2,add,linear,both,core_linear_C0.001_halfNone,both_linear_C0.001_halfNone,0.004568,0.004999,0.008237,0.000548,0.004606,0.006078,0.002769,0.006116
3,remove_multilevel,linear,style,both_linear_C0.001_halfNone,style_linear_C0.001_halfNone,-0.006362,-0.004514,-0.013180,-0.000359,-0.005025,-0.005538,-0.004351,-0.005542
4,remove_style,linear,multilevel,both_linear_C0.001_halfNone,multilevel_linear_C0.001_halfNone,-0.001794,-0.000476,0.000000,-0.000082,-0.000838,-0.000671,-0.003560,-0.000481
5,add,linear,multilevel,core_linear_C0.001_half3.0,multilevel_linear_C0.001_half3.0,0.007015,0.003852,0.009061,-0.000766,0.003769,0.004926,0.009098,0.005057
6,add,linear,style,core_linear_C0.001_half3.0,style_linear_C0.001_half3.0,0.002121,0.000553,0.004942,0.000422,0.001256,0.000544,0.001582,0.000623
7,add,linear,both,core_linear_C0.001_half3.0,both_linear_C0.001_half3.0,0.007015,0.004300,0.012356,-0.000486,0.006281,0.005459,0.005142,0.005505
8,remove_multilevel,linear,style,both_linear_C0.001_half3.0,style_linear_C0.001_half3.0,-0.004894,-0.003748,-0.007414,0.000909,-0.005025,-0.004914,-0.003560,-0.004882
9,remove_style,linear,multilevel,both_linear_C0.001_half3.0,multilevel_linear_C0.001_half3.0,0.000000,-0.000448,-0.003295,-0.000280,-0.002513,-0.000533,0.003956,-0.000448


,design,components,gamma_multiplier,C,reference,candidate,pooled_accuracy_gain,pooled_loss_reduction,2020_accuracy_gain,2020_loss_reduction,2021_accuracy_gain,2021_loss_reduction,2022_accuracy_gain,2022_loss_reduction
0,linear,0,0.0,0.003,r5_linear_logistic_C0.003,r5_linear_squared_hinge_C0.003,-0.002936,0.000218,-0.000824,0.000578,-0.000838,-0.000605,-0.005934,0.000823
1,linear,0,0.0,0.030,r5_linear_logistic_C0.03,r5_linear_squared_hinge_C0.03,-0.001958,-0.000735,-0.002471,0.000857,0.001256,-0.000885,-0.004747,-0.001359
2,context,0,0.0,0.003,r5_context_logistic_C0.003,r5_context_squared_hinge_C0.003,0.000163,0.000339,-0.003295,0.000426,0.002094,-0.000739,0.000000,0.001316
3,context,0,0.0,0.030,r5_context_logistic_C0.03,r5_context_squared_hinge_C0.03,0.002447,-0.000967,0.006590,0.000663,0.004606,-0.001138,-0.001582,-0.001589
4,rff,256,0.5,0.003,r5_rff256_g0.5_logistic_C0.003,r5_rff256_g0.5_squared_hinge_C0.003,-0.000816,0.000063,-0.005766,0.000162,0.000838,-0.000755,0.000000,0.000790
5,rff,256,0.5,0.030,r5_rff256_g0.5_logistic_C0.03,r5_rff256_g0.5_squared_hinge_C0.03,-0.001794,-0.001940,0.001647,-0.000016,-0.001256,-0.001708,-0.003956,-0.003083
6,rff,256,2.0,0.003,r5_rff256_g2_logistic_C0.003,r5_rff256_g2_squared_hinge_C0.003,-0.001142,-0.000367,0.004942,-0.000189,0.000000,-0.001088,-0.005142,0.000229
7,rff,256,2.0,0.030,r5_rff256_g2_logistic_C0.03,r5_rff256_g2_squared_hinge_C0.03,0.002936,-0.003889,0.013180,-0.002058,-0.000419,-0.004067,0.001187,-0.004601
8,rff,512,0.5,0.003,r5_rff512_g0.5_logistic_C0.003,r5_rff512_g0.5_squared_hinge_C0.003,0.000000,0.000086,-0.002471,0.000176,-0.001675,-0.000731,0.002769,0.000814
9,rff,512,0.5,0.030,r5_rff512_g0.5_logistic_C0.03,r5_rff512_g0.5_squared_hinge_C0.03,0.001142,-0.001537,0.003295,0.000291,0.000000,-0.001437,0.001187,-0.002511


## 训练期家族选择与全部候选

家族先过log-loss护栏，再按accuracy选择。未满足护栏的候选仍保存在完整CV表。

In [3]:
display(family_table(data))
display(data['cv'].sort_values(['accuracy','mean_log_loss'],ascending=[False,True]))

,family,model,CV_accuracy,CV_log_loss,2023_accuracy,2023_log_loss
17,r5_rff_squared_hinge,r5_rff512_g2_squared_hinge_C0.03,0.670962,0.602519,0.656105,0.614894
16,r5_rff_logistic,r5_rff512_g2_logistic_C0.03,0.668515,0.598554,0.659969,0.608984
2,both_spline,both_spline_C0.01_half5.0,0.668026,0.598466,0.656491,0.607241
14,r5_linear_logistic,r5_linear_logistic_C0.003,0.667374,0.599464,0.653014,0.607572
1,both_linear,both_linear_C0.1_half3.0,0.667047,0.599516,0.658037,0.608254
7,new_catboost,both_cat_d4_it1200,0.666558,0.602035,0.652628,0.611976
5,multilevel_linear,multilevel_linear_C0.01_half3.0,0.666395,0.599400,0.654946,0.607615
12,r5_context_squared_hinge,r5_context_squared_hinge_C0.03,0.666232,0.599535,0.659583,0.611118
6,multilevel_spline,multilevel_spline_C0.03_half3,0.665742,0.598557,0.661128,0.608014
13,r5_existing_spline,r5_existing_odd_spline_C0.01_context1_half3.0,0.665742,0.598764,0.658423,0.607506


,name,family,accuracy,mean_log_loss,2020_accuracy,2020_log_loss,2021_accuracy,2021_log_loss,2022_accuracy,2022_log_loss
64,r5_rff512_g2_squared_hinge_C0.03,r5_rff_squared_hinge,0.670962,0.602519,0.681219,0.608407,0.659548,0.609559,0.676820,0.593041
63,r5_rff512_g2_squared_hinge_C0.003,r5_rff_squared_hinge,0.670147,0.599092,0.673806,0.606070,0.661642,0.605869,0.676424,0.589339
62,r5_rff512_g2_logistic_C0.03,r5_rff_logistic,0.668515,0.598554,0.668863,0.606725,0.662060,0.604904,0.674446,0.588631
60,r5_rff512_g0.5_squared_hinge_C0.03,r5_rff_squared_hinge,0.668515,0.599954,0.668040,0.606372,0.659966,0.606183,0.676820,0.590989
38,both_spline_C0.01_half5.0,both_spline,0.668026,0.598466,0.659802,0.607259,0.662898,0.604224,0.676820,0.588805
58,r5_rff512_g0.5_logistic_C0.03,r5_rff_logistic,0.667374,0.598417,0.664745,0.606663,0.659966,0.604746,0.675633,0.588478
41,r5_linear_logistic_C0.003,r5_linear_logistic,0.667374,0.599464,0.662273,0.606539,0.658291,0.604835,0.678402,0.590994
32,both_linear_C0.1_half3.0,both_linear,0.667047,0.599516,0.667216,0.607427,0.660385,0.604503,0.673259,0.591007
56,r5_rff256_g2_squared_hinge_C0.03,r5_rff_squared_hinge,0.666721,0.602864,0.676277,0.609955,0.659548,0.608747,0.668908,0.593902
40,both_linear_C0.01_half5.0,both_linear,0.666558,0.598842,0.659802,0.606684,0.659548,0.604429,0.676424,0.589797


## 冻结决定与配对不确定性

纠正旧错和新增错误均需计入净增益；日期块区间没有修复多轮模型搜索偏差。

In [4]:
display(data['frozen'])
display(pd.DataFrame([{'model':n,**v['accuracy']} for n,v in data['summary']['comparisons_vs_round4'].items()]))
assert data['summary']['final_test_evaluated'] is False
print('2024–2025 final holdout remains unopened.')

{'round': 5,
 'recommendation': 'previous_probability_recommendation',
 'probability_reference': 'previous_probability_recommendation',
 'procedure_gate': {'procedure': 'retain_round4',
  'early_selection': {'choice': 'accuracy_grid_blend',
   'best_single': 'r5_rff512_g2_squared_hinge_C0.03',
   'selected_by_family': {'all_inputs': 'all_inputs_spline_C0.01',
    'both_linear': 'both_linear_C0.1_halfNone',
    'both_spline': 'both_spline_C0.01_half5.0',
    'core_linear': 'core_linear_C0.1_halfNone',
    'core_spline': 'core_spline_C0.03_half3',
    'multilevel_linear': 'multilevel_linear_C0.1_halfNone',
    'multilevel_spline': 'multilevel_spline_C0.03_half3',
    'new_catboost': 'both_cat_d4_it1200',
    'new_histboost': 'both_hist_l15',
    'previous_accuracy_comparator': 'previous_accuracy_comparator',
    'previous_exploratory_best_loss': 'previous_exploratory_best_loss',
    'previous_probability_recommendation': 'previous_probability_recommendation',
    'r5_context_logistic': '

,model,accuracy_improvement,net_correct,corrected,newly_wrong,ci95,date_blocks,repetitions,interpretation
0,all_inputs_spline_C0.01,0.003864,10,76,66,"[-0.004383, 0.012706]",35,2000,Exploratory paired date-block interval; no cor...
1,both_cat_d4_it1200,-0.002705,-7,101,108,"[-0.012948, 0.008807]",35,2000,Exploratory paired date-block interval; no cor...
2,both_hist_l7,-0.000386,-1,105,106,"[-0.010968, 0.010786]",35,2000,Exploratory paired date-block interval; no cor...
3,both_linear_C0.1_half3.0,0.002705,7,84,77,"[-0.004122, 0.009484]",35,2000,Exploratory paired date-block interval; no cor...
4,both_spline_C0.01_half5.0,0.001159,3,75,72,"[-0.006604, 0.009241]",35,2000,Exploratory paired date-block interval; no cor...
5,core_linear_C0.1_halfNone,-0.000773,-2,43,45,"[-0.007458, 0.005389]",35,2000,Exploratory paired date-block interval; no cor...
6,core_spline_C0.03_half3,-0.001159,-3,12,15,"[-0.004945, 0.002582]",35,2000,Exploratory paired date-block interval; no cor...
7,multilevel_linear_C0.01_half3.0,-0.000386,-1,70,71,"[-0.008571, 0.007695]",35,2000,Exploratory paired date-block interval; no cor...
8,multilevel_spline_C0.03_half3,0.005796,15,82,67,"[-0.001312, 0.012483]",35,2000,Exploratory paired date-block interval; no cor...
9,r5_context_logistic_C0.003,-0.001159,-3,64,67,"[-0.009115, 0.007211]",35,2000,Exploratory paired date-block interval; no cor...


2024–2025 final holdout remains unopened.
